# 05 · Test the pre-registered hypotheses
Locked on 18 November 2026, before any market data was downloaded (`input/ideas/oddsgap-plan.md`).
- **H1:** the median pair gap is at least 2 percentage points.
- **H2:** gaps are smaller in the last 24 hours than more than 72 hours out.
- **H3:** neither platform is more accurate (Brier score) 24 hours before the close.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "data"
comp = pd.read_parquet(OUT / "comparable.parquet")
gaps = pd.read_parquet(OUT / "pair_gaps.parquet")
print(len(gaps), "pairs ·", len(comp), "comparable points")

1457 pairs · 42219 comparable points


### The bootstrap
Resample the pairs 10,000 times (seed 42) and keep the middle 90% of the statistic.

In [2]:
def boot(values, stat=np.median, n=10_000, seed=42):
    v = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    sims = stat(rng.choice(v, (n, len(v))), axis=1)
    lo, hi = np.percentile(sims, [5, 95])
    return {"value": float(stat(v)), "low": float(lo), "high": float(hi), "n": int(len(v))}

### H1: how far apart are the two prices?

In [3]:
h1 = boot(gaps["gap_pp"])
h1["verdict"] = "supported" if h1["low"] >= 2 else "rejected" if h1["high"] < 2 else "inconclusive"
print(f"median pair gap {h1['value']:.2f} pp (90% range {h1['low']:.2f}–{h1['high']:.2f}) over {h1['n']} pairs → H1 {h1['verdict']}")

median pair gap 0.75 pp (90% range 0.75–1.00) over 1457 pairs → H1 rejected


### H2: do gaps close near the end?
For pairs with points both more than 72 hours out and in the final 24 hours: far gap minus near gap.

In [4]:
far = comp[comp["hours_to_close"] > 72].groupby("pair_id")["gap_pp"].median()
near = comp[comp["hours_to_close"] <= 24].groupby("pair_id")["gap_pp"].median()
both = pd.concat([far.rename("far"), near.rename("near")], axis=1).dropna()
h2 = boot(both["far"] - both["near"])
h2.update({"far_median": float(both["far"].median()), "near_median": float(both["near"].median())})
h2["verdict"] = "supported" if h2["low"] > 0 else "rejected"
print(f"{h2['n']} pairs: far {h2['far_median']:.2f} pp → near {h2['near_median']:.2f} pp; change {h2['value']:.2f} "
      f"(90% range {h2['low']:.2f}–{h2['high']:.2f}) → H2 {h2['verdict']}")

510 pairs: far 1.00 pp → near 0.00 pp; change 1.00 (90% range 0.75–1.00) → H2 supported


### H3: which platform was more accurate?
For each pair: the last comparable point at least 24 hours before the close. Brier score = (price − outcome)², lower is better.
Pairs that settled differently on the two platforms are left out (the "outcome" isn't one thing there).

In [5]:
early = comp[(comp["hours_to_close"] >= 24) & (~comp["settled_differently"])].sort_values("ts").groupby("pair_id").tail(1)
early = early.assign(brier_k=(early["kalshi"] - early["outcome"]) ** 2, brier_p=(early["polymarket"] - early["outcome"]) ** 2)
h3 = boot(early["brier_k"] - early["brier_p"], stat=np.mean)
h3.update({"brier_kalshi": float(early["brier_k"].mean()), "brier_polymarket": float(early["brier_p"].mean())})
h3["verdict"] = "rejected" if (h3["low"] > 0 or h3["high"] < 0) else "supported"
h3["better"] = None if h3["verdict"] == "supported" else ("polymarket" if h3["value"] > 0 else "kalshi")
print(f"{h3['n']} pairs: Brier Kalshi {h3['brier_kalshi']:.4f}, Polymarket {h3['brier_polymarket']:.4f}; "
      f"difference {h3['value']:.4f} (90% range {h3['low']:.4f}–{h3['high']:.4f}) → H3 {h3['verdict']}")

1456 pairs: Brier Kalshi 0.2256, Polymarket 0.2254; difference 0.0003 (90% range -0.0002–0.0007) → H3 supported


### Reported, not deciding: by category

In [6]:
by_cat = []
for c, g in gaps.groupby("category"):
    b = boot(g["gap_pp"])
    e = early[early["category"] == c]
    by_cat.append({"category": c, "pairs": int(len(g)), "gap_pp": b["value"], "low": b["low"], "high": b["high"],
                   "share_over_2pp": float((g["gap_pp"] >= 2).mean() * 100), "kalshi_higher_pp": float(g["diff_pp"].median()),
                   "brier_kalshi": float(e["brier_k"].mean()) if len(e) else None,
                   "brier_polymarket": float(e["brier_p"].mean()) if len(e) else None, "brier_pairs": int(len(e))})
by_cat = pd.DataFrame(by_cat).sort_values("pairs", ascending=False)
by_cat.round(3)

,category,pairs,gap_pp,low,high,share_over_2pp,kalshi_higher_pp,brier_kalshi,brier_polymarket,brier_pairs
4,NHL,535,1.00,0.75,1.0,7.103,0.0,0.243,0.243,535
2,NBA,497,0.75,0.75,1.0,2.817,0.0,0.214,0.214,497
1,MLB,270,0.75,0.75,1.0,4.074,0.0,0.242,0.243,269
3,NFL,114,0.50,0.50,0.5,1.754,0.0,0.235,0.234,114
0,Fed,41,1.00,1.00,1.5,26.829,0.0,0.006,0.005,41


### Reported, not deciding: who moves first?
Within each pair, take each platform's 3-hour price change between consecutive grid times. If Kalshi's change today
predicts Polymarket's change at the next step better than the other way round, Kalshi tends to move first.

In [7]:
c = comp.sort_values(["pair_id", "ts"]).copy()
c["dk"] = c.groupby("pair_id")["kalshi"].diff()
c["dp"] = c.groupby("pair_id")["polymarket"].diff()
c["dk_next"] = c.groupby("pair_id")["dk"].shift(-1)
c["dp_next"] = c.groupby("pair_id")["dp"].shift(-1)
c["step_ok"] = c.groupby("pair_id")["ts"].diff() == 10800          # only consecutive 3-hour steps
ok = c[c["step_ok"] & c.groupby("pair_id")["step_ok"].shift(-1).fillna(False).astype(bool)].dropna(subset=["dk", "dp", "dk_next", "dp_next"])
lead = {
    "same_step": float(ok["dk"].corr(ok["dp"])),
    "kalshi_then_poly": float(ok["dk"].corr(ok["dp_next"])),
    "poly_then_kalshi": float(ok["dp"].corr(ok["dk_next"])),
    "steps": int(len(ok)),
}
lead

{'same_step': 0.5002625929813452,
 'kalshi_then_poly': 0.005796426623020317,
 'poly_then_kalshi': -0.017876342648696704,
 'steps': 30063}

### Reported, not deciding: pairs that settled differently

In [8]:
pairs = pd.read_parquet(OUT / "pairs.parquet")
mismatch = pairs[pairs["settled_differently"]][["category", "date", "kalshi_ticker", "poly_slug", "kalshi_result", "poly_result", "close_gap_hours"]]
mismatch

,category,date,kalshi_ticker,poly_slug,kalshi_result,poly_result,close_gap_hours
1,NBA,2025-10-10,KXNBAGAME-25OCT10ORLPHI-ORL,nba-orl-phi-2025-10-10,0,1,3.884759
2,NBA,2025-10-10,KXNBAGAME-25OCT10BOSTOR-BOS,nba-bos-tor-2025-10-10,1,0,3.986995
2558,MLB,2025-07-04,KXMLBGAME-25JUL04HOULAD-HOU,mlb-hou-lad-2025-07-04,1,0,1.796325
6030,MLB,2026-09-25,KXMLBGAME-26SEP251905BALNYY-BAL,mlb-bal-nyy-2026-09-25,0,1,3.198056
6052,MLB,2026-09-26,KXMLBGAME-26SEP261915BALNYY-BAL,mlb-bal-nyy-2026-09-26,1,0,4.289167


### Reported, not deciding: average gaps and identical prices
Both platforms quote in steps of half a cent or a cent, so per-pair medians land on round steps (0.5, 0.75, 1.0 …).
The average gap per pair and the share of moments with *identical* prices give a finer picture.

In [9]:
pair_mean = comp.groupby("pair_id")["gap_pp"].mean()
detail = {"mean_gap_per_pair_median": float(pair_mean.median()), "mean_gap_all_points": float(comp["gap_pp"].mean()),
          "identical_share": float((comp["gap_pp"] < 1e-9).mean() * 100), "over_5pp_share": float((comp["gap_pp"] >= 5).mean() * 100)}
detail

{'mean_gap_per_pair_median': 1.0833333333333341,
 'mean_gap_all_points': 1.3708330372581066,
 'identical_share': 24.586086832942513,
 'over_5pp_share': 4.725360619626234}

### Check: are the two series lined up in time?
The plan flagged a risk: Polymarket's 3-hour timestamps might mark the start of each step, not the end. If so, shifting its
series by one step would bring the two prices closer. I try shifts of −2 to +2 steps and keep the result.

In [10]:
prices = pd.read_parquet(OUT / "prices.parquet")
k = prices[prices["platform"] == "kalshi"][["pair_id", "ts", "price"]].rename(columns={"price": "k"})
pm = prices[prices["platform"] == "polymarket"][["pair_id", "ts", "price"]].rename(columns={"price": "p"})
k = k[k["pair_id"].isin(gaps["pair_id"])]
shifts = []
for step in [-2, -1, 0, 1, 2]:
    m = k.merge(pm.assign(ts=pm["ts"] + step * 10800), on=["pair_id", "ts"])
    m = m[m["k"].between(0.03, 0.97) & m["p"].between(0.03, 0.97)]
    shifts.append({"shift_steps": step, "points": int(len(m)), "mean_gap_pp": float((m["k"] - m["p"]).abs().mean() * 100)})
shifts = pd.DataFrame(shifts)
best = int(shifts.loc[shifts["mean_gap_pp"].idxmin(), "shift_steps"])
print("best alignment:", best, "steps (0 means the timestamps line up as assumed)")
shifts.round(3)

best alignment: 0 steps (0 means the timestamps line up as assumed)


,shift_steps,points,mean_gap_pp
0,-2,41792,1.850
1,-1,42423,1.727
2,0,42860,1.361
3,1,42750,1.819
4,2,42577,1.896


### Save

In [11]:
results = {"h1": h1, "h2": h2, "h3": h3, "lead": lead, "detail": detail,
           "alignment": {"best_shift": best, "table": shifts.to_dict("records")}, "pre_registered": "2026-11-18",
           "pairs_tested": int(len(gaps)), "points": int(len(comp)),
           "by_category": by_cat.to_dict("records"),
           "mismatches": mismatch.assign(date=mismatch["date"].astype(str)).to_dict("records")}
(OUT / "results.json").write_text(json.dumps(results, indent=1, default=float))
print(json.dumps({k: results[k] for k in ("h1", "h2", "h3")}, indent=1))

{
 "h1": {
  "value": 0.7500000000000007,
  "low": 0.7500000000000007,
  "high": 0.9999999999999898,
  "n": 1457,
  "verdict": "rejected"
 },
 "h2": {
  "value": 0.9999999999999996,
  "low": 0.7500000000000091,
  "high": 1.0000000000000009,
  "n": 510,
  "far_median": 1.0000000000000009,
  "near_median": 0.0,
  "verdict": "supported"
 },
 "h3": {
  "value": 0.000263980769230772,
  "low": -0.00015774405048076765,
  "high": 0.000680918063186817,
  "n": 1456,
  "brier_kalshi": 0.2256150412087912,
  "brier_polymarket": 0.2253510604395604,
  "verdict": "supported",
  "better": null
 }
}
